In [1]:
import sqlite3

# Re-create database setup in memory / script environment to ensure full execution
conn = sqlite3.connect("order_management.db")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

cursor.executescript("""
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    phone TEXT,
    city TEXT
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name TEXT NOT NULL,
    category TEXT,
    price REAL NOT NULL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    status TEXT DEFAULT 'Pending',
    total_amount REAL,
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id) ON DELETE CASCADE
);

CREATE TABLE order_items (
    item_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price REAL NOT NULL,
    FOREIGN KEY (order_id) REFERENCES orders (order_id) ON DELETE CASCADE,
    FOREIGN KEY (product_id) REFERENCES products (product_id)
);

INSERT INTO customers (first_name, last_name, email, phone, city) VALUES
('John', 'Doe', 'john.doe@email.com', '555-0192', 'New York'),
('Jane', 'Smith', 'jane.smith@email.com', '555-0193', 'Los Angeles'),
('Robert', 'Johnson', 'robert.j@email.com', '555-0194', 'Chicago'),
('Emily', "Davis", 'emily.davis@email.com', '555-0195', 'Houston');

INSERT INTO products (product_name, category, price) VALUES
('Laptop', 'Electronics', 1200.00),
('Wireless Mouse', 'Electronics', 25.50),
('Mechanical Keyboard', 'Electronics', 85.00),
('Office Chair', 'Furniture', 150.00);

INSERT INTO orders (customer_id, order_date, status, total_amount) VALUES
(1, '2026-09-01', 'Completed', 1225.50),
(2, '2026-09-02', 'Shipped', 150.00),
(1, '2026-09-03', 'Processing', 85.00);

INSERT INTO order_items (order_id, product_id, quantity, unit_price) VALUES
(1, 1, 1, 1200.00),
(1, 2, 1, 25.50),
(2, 4, 1, 150.00),
(3, 3, 1, 85.00);
""")
conn.commit()

# Problem 1
print("--- P1 ---")
print("Retrieve all details for all registered customers in the database.")
cursor.execute("SELECT * FROM customers;")
# print(cursor.fetchall())
data=cursor.fetchall()
for row in data:
    print(row)

# Problem 2
print("--- P2 ---")
print("Retrieve a list of all orders along with the corresponding customer details (ID, first name, last name, order ID, order date, order status, and total order amount) using an INNER JOIN.")
cursor.execute("""
SELECT c.customer_id, c.first_name, c.last_name, o.order_id, o.order_date, o.status, o.total_amount
FROM customers c
INNER JOIN orders o ON c.customer_id = o.customer_id;
""")
# print(cursor.fetchall())
data=cursor.fetchall()
for row in data:
    print(row)

# Problem 3
print("--- P3 ---")
print("Calculate the total amount spent by each customer across all their orders, showing their customer ID, first name, last name, and total spent amount.")
cursor.execute("""
SELECT c.customer_id, c.first_name, c.last_name, SUM(o.total_amount) AS total_spent
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id;
""")
# print(cursor.fetchall())
data=cursor.fetchall()
for row in data:
    print(row)

# Problem 4
print("--- P4 ---")
print("Retrieve detailed line-item information for every order, including the order ID, customer's first and last name, product name, quantity purchased, and unit price.")
cursor.execute("""
SELECT o.order_id, c.first_name, c.last_name, p.product_name, oi.quantity, oi.unit_price
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON oi.product_id = p.product_id;
""")
# print(cursor.fetchall())
data=cursor.fetchall()
for row in data:
    print(row)

# Problem 5
print("--- P5 ---")
print("Find all customers who have not placed any orders yet, returning their customer ID, first name, last name, and email address.")
cursor.execute("""
SELECT c.customer_id, c.first_name, c.last_name, c.email
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL;
""")
# print(cursor.fetchall())
data=cursor.fetchall()
for row in data:
    print(row)

# Problem 6
print("--- P6 ---")
print("Calculate the total revenue/sales generated by product category by multiplying the quantity by unit price for each item sold.")
cursor.execute("""
SELECT p.category, SUM(oi.quantity * oi.unit_price) AS total_sales
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category;
""")
# print(cursor.fetchall())
data=cursor.fetchall()
for row in data:
    print(row)

conn.close()

--- P1 ---
Retrieve all details for all registered customers in the database.
(1, 'John', 'Doe', 'john.doe@email.com', '555-0192', 'New York')
(2, 'Jane', 'Smith', 'jane.smith@email.com', '555-0193', 'Los Angeles')
(3, 'Robert', 'Johnson', 'robert.j@email.com', '555-0194', 'Chicago')
(4, 'Emily', 'Davis', 'emily.davis@email.com', '555-0195', 'Houston')
--- P2 ---
Retrieve a list of all orders along with the corresponding customer details (ID, first name, last name, order ID, order date, order status, and total order amount) using an INNER JOIN.
(1, 'John', 'Doe', 1, '2026-09-01', 'Completed', 1225.5)
(2, 'Jane', 'Smith', 2, '2026-09-02', 'Shipped', 150.0)
(1, 'John', 'Doe', 3, '2026-09-03', 'Processing', 85.0)
--- P3 ---
Calculate the total amount spent by each customer across all their orders, showing their customer ID, first name, last name, and total spent amount.
(1, 'John', 'Doe', 1310.5)
(2, 'Jane', 'Smith', 150.0)
--- P4 ---
Retrieve detailed line-item information for every orde